In [2]:
import requests
import pandas as pd
import time
import os

In [19]:
app_ids = [
    730, 570, 440, 578080, 271590,
    1172470, 1085660, 381210, 1091500, 1245620,
    1938090, 252490, 1599340, 413150, 814380,
    292030, 553850, 236390, 394360, 1086940
]

In [16]:
url = f"https://store.steampowered.com/appreviews/{730}"
response = requests.get(url, params = {
        "json": 1,
        "filter": "summary",
    }, timeout=15)
data = response.json()
print(data.keys())

dict_keys(['success', 'query_summary', 'reviews', 'cursor'])


In [17]:
def get_review_summary(app_id):
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    
    params = {
        "json": 1,
        "filter": "summary",
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
        
        query_summary = data.get("query_summary", {})
        
        total_positive = query_summary.get("total_positive")
        total_negative = query_summary.get("total_negative")
        
        total_reviews = None
        positive_ratio = None
        
        if total_positive is not None and total_negative is not None:
            total_reviews = total_positive + total_negative
            if total_reviews > 0:
                positive_ratio = total_positive / total_reviews
        
        return {
            "app_id": app_id,
            "review_score_desc": query_summary.get("review_score_desc"),
            "review_score": query_summary.get("review_score"),
            "total_positive": total_positive,
            "total_negative": total_negative,
            "total_reviews": total_reviews,
            "positive_ratio": positive_ratio
        }
    
    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return None

In [20]:
review_records = []

for app_id in app_ids:
    row = get_review_summary(app_id)
    if row is not None:
        review_records.append(row)
    time.sleep(1)

df_reviews = pd.DataFrame(review_records)
df_reviews.head()

,app_id,review_score_desc,review_score,total_positive,total_negative,total_reviews,positive_ratio
0,730,Very Positive,8,1232060,207437,1439497,0.855896
1,570,Very Positive,8,5428,676,6104,0.889253
2,440,Mostly Positive,6,20572,6423,26995,0.762067
3,578080,Mixed,5,176620,101855,278475,0.634240
4,271590,Very Positive,8,486766,92305,579071,0.840598


In [5]:
print(df_reviews.shape)
print(df_reviews.info())
print(df_reviews.isnull().sum())

(20, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   app_id             20 non-null     int64  
 1   review_score_desc  20 non-null     object 
 2   review_score       20 non-null     int64  
 3   total_positive     20 non-null     int64  
 4   total_negative     20 non-null     int64  
 5   total_reviews      20 non-null     int64  
 6   positive_ratio     20 non-null     float64
dtypes: float64(1), int64(5), object(1)
memory usage: 1.2+ KB
None
app_id               0
review_score_desc    0
review_score         0
total_positive       0
total_negative       0
total_reviews        0
positive_ratio       0
dtype: int64


In [9]:
os.makedirs("../data/raw", exist_ok=True)
df_reviews.to_csv("../data/raw/steam_reviews_batch1.csv", index=False)

print("Saved raw reviews batch.")

Saved raw reviews batch.


In [7]:
df_reviews_clean = df_reviews.copy()

df_reviews_clean.columns = df_reviews_clean.columns.str.strip().str.lower()
df_reviews_clean = df_reviews_clean.drop_duplicates(subset="app_id")

df_reviews_clean["review_score"] = pd.to_numeric(df_reviews_clean["review_score"], errors="coerce")
df_reviews_clean["total_positive"] = pd.to_numeric(df_reviews_clean["total_positive"], errors="coerce")
df_reviews_clean["total_negative"] = pd.to_numeric(df_reviews_clean["total_negative"], errors="coerce")
df_reviews_clean["total_reviews"] = pd.to_numeric(df_reviews_clean["total_reviews"], errors="coerce")
df_reviews_clean["positive_ratio"] = pd.to_numeric(df_reviews_clean["positive_ratio"], errors="coerce")

os.makedirs("../data/cleaned", exist_ok=True)
df_reviews_clean.to_csv("../data/cleaned/steam_reviews_clean.csv", index=False)

print("Saved cleaned reviews.")

Saved cleaned reviews.
